# COMP - 2040: Final Project  
**Name:** Troy Dela Rosa  
**SID#** 0213352  
**Date** April 11, 2026  
**Instructor:** Chris Mac  

## 1. Introduction

### How Instructor Feedback Changed This Project

In weeks 12 and 13 I focused on downtown Winnipeg business license activity from 2023 to 2025. The data came from the City of Winnipeg open-data portal, and the plan was to look at whether license patterns showed a measurable decline.

After the week 13 check-in, Chris pointed out that a three-year window is too narrow to tell whether what I was seeing was a real structural trend or just short-term noise. He suggested pushing the timeline back as far as the data would allow.

That feedback changed the project in a big way. I called 311 and submitted a formal data request through the City's website, but the historical license records I was hoping for were not available in time. Instead I expanded the analysis by building and collecting additional datasets from public sources — CBRE vacancy reports, Downtown BIZ snapshots, StatCan census data, and local news reporting — to piece together a longer-term picture stretching back to 2010.

The result is a project that no longer just asks *"is downtown declining?"* but instead tries to understand *how, when, and in what way* downtown Winnipeg has been changing — and whether the $2.3 billion in tracked investment represents recovery, or something structurally different.

### Central Argument

Downtown Winnipeg is not recovering in the traditional sense. It is converting from a retail-and-office-anchored model to a residential and institutional one. The investment does not attempt to restore what was lost — it bets on a different urban form.

### Three Analytical Questions

1. **How has the balance of Growth vs. Transition events shifted across the three phases of downtown development?**
2. **Has downtown business activity declined — and which sectors were hit hardest?**
3. **Can we predict the delivery status of a housing project (Completed / Under Construction / Planned) based on its characteristics?**

## 2. Data Sources and Research Design

### What datasets am I working with?

This project uses several datasets I built and collected from public sources:

- **`downtown_wpg_gantt_sourced_2026.csv`** — 28 major structural events in Downtown Winnipeg between 2010 and 2026, including construction projects, retail closures, and policy decisions. Each row records a start date, end date, event category, and a source provenance rating.
- **`housing_pipeline_2026.csv`** — residential development projects in the downtown pipeline, with unit count ranges, delivery status, and confidence levels.
- **`Business_Licenses_20260404.csv`** — City of Winnipeg open data: downtown business license records from 2021 to 2026.
- **`Winnipeg_Synthetic_License_Model_2010_2024.xlsx`** — A calibrated synthetic model that estimates downtown business activity from 2010 to 2020, where official open data is unavailable.
- **`downtown_wpg_sources_2026.csv`** — A source provenance registry documenting every figure with its source, confidence score, and usage type.

### Where does the data come from?

The business license data comes directly from the City of Winnipeg Open Data portal. The structural events and housing pipeline datasets were built by me from public sources: CBRE market reports, Downtown Winnipeg BIZ annual snapshots, and CBC Manitoba news reporting. The full source registry documents 53 sources.

### Data Layer Structure

Each section is labeled so the reader knows how reliable every number is:

| Layer | Label | Meaning |
|---|---|---|
| **L1 — Observed** | `[OBSERVED]` | Directly confirmed from primary sources |
| **L2 — Reconstructed** | `[RECONSTRUCTED]` | Calibrated estimates filling known data gaps |
| **L3 — Inferred** | `[INFERRED]` | Patterns derived from L1/L2, not direct measurements |
| **L4 — Scenario** | `[SCENARIO]` | Model outputs under stated assumptions |

## 3. Setup

In [11]:
# standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# scikit-learn imports for prediction section
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# dark chart style — all charts inherit this
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor':   '#161921',
    'axes.edgecolor':   '#2a2e3a',
    'axes.labelcolor':  '#e0ddd5',
    'text.color':       '#e0ddd5',
    'xtick.color':      '#8a8780',
    'ytick.color':      '#8a8780',
    'grid.color':       '#2a2e3a',
    'font.family':      'sans-serif',
    'font.size':        11,
    'figure.dpi':       120
})

# colour constants
ACCENT    = '#c9a44a'   # gold
GROWTH_C  = '#4e9a6e'   # green
TRANS_C   = '#5a7bbf'   # blue
DECLINE_C = '#bf4e4e'   # red

PHASE_COLORS = {
    'Pre-2015 establishment':    '#6e8a9e',
    '2015\u20132020 transition': DECLINE_C,
    '2020+ restructuring':       GROWTH_C,
}

TYPE_COLORS = {
    'Growth':         '#4e9a6e',
    'Infrastructure': '#5a7bbf',
    'Transition':     '#bf4e4e',
    'Adaptive Reuse': '#c9a44a',
    'Policy':         '#8a6e9a',
}

print("Ready.")

Ready.


### Helper Module — New Function for Week 14

Weeks 12–13 already had three helper functions in `src/helpers.py`:
- `clean_column_names()` — lowercase + underscore formatting
- `create_is_closed()` — flags closed/cancelled/ceased licenses
- `filter_downtown()` — keeps only downtown Community Characterization Area

For week 14 I added a fourth function that assigns each gantt event to its development phase based on start year. I'm defining it here so the notebook is self-contained, but it also lives in `helpers.py`.

## 4. Data Overview

Before cleaning or analysing anything, I want to understand the basic shape of each dataset — how many rows, what the columns are, and where the missing values are.

In [1]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\troyd\_Troy\Term2-DSML\Python Ess- Comp 2040\final_project\comp-2040-project-3\notebooks
['week12.ipynb', 'week13.ipynb', 'week14.ipynb']


### 4.1 Gantt Sourced Dataset

In [4]:
import pandas as pd
import os

print("Current working directory:")
print(os.getcwd())
print()

# ── Gantt dataset: basic shape and types ──
gantt_raw = pd.read_csv('../data/downtown_wpg_gantt_sourced_2026.csv')

print("=== GANTT DATASET ===")
print(f"Shape: {gantt_raw.shape}\n")

print("Columns and types:")
print(gantt_raw.dtypes.to_string())
print()

print("Missing values per column:")
print(gantt_raw.isnull().sum().to_string())
print()

print("First 3 rows:")
print(
    gantt_raw[
        ['task_name', 'category', 'start_date', 'end_date', 'status', 'source_quality']
    ].head(3).to_string(index=False)
)

Current working directory:
c:\Users\troyd\_Troy\Term2-DSML\Python Ess- Comp 2040\final_project\comp-2040-project-3\notebooks

=== GANTT DATASET ===
Shape: (30, 12)

Columns and types:
task_name              object
category               object
type                   object
start_date             object
end_date               object
status                 object
row_type               object
source_quality         object
biz_fallback             bool
primary_source_id      object
secondary_source_id    object
source_note            object

Missing values per column:
task_name               0
category                0
type                    0
start_date              0
end_date               11
status                  0
row_type                0
source_quality          0
biz_fallback            0
primary_source_id       0
secondary_source_id     6
source_note             0

First 3 rows:
                       task_name       category start_date   end_date    status source_quality
 RBC C

In [5]:
print(gantt_raw['source_quality'].value_counts())
print()
print(gantt_raw['row_type'].value_counts())

source_quality
strong      12
weak         9
moderate     9
Name: count, dtype: int64

row_type
discrete     26
composite     4
Name: count, dtype: int64


#### 5 observations about the Gantt dataset:

1. The dataset contains **30 rows**, which confirms that it is a small, curated project-level dataset rather than a large raw administrative file. Most rows represent discrete structural events, but a few represent broader ongoing processes.

2. Both **`start_date`** and **`end_date`** are stored as text (`object`) rather than datetime values, so they need to be converted before any duration calculation or timeline plotting.

3. The **`end_date`** field has **11 missing values**, which likely correspond to projects or processes that are still ongoing. These missing values need to be handled explicitly so in-progress projects are not dropped from the analysis.

4. The dataset includes built-in source provenance. Of the 30 rows, **12 are rated strong**, **9 moderate**, and **9 weak** on `source_quality`, which makes it possible to distinguish more strongly sourced events from weaker ones in later interpretation.

5. The **`row_type`** field shows that **26 rows are discrete events** and **4 are composite rows**. This matters because composite process rows can be useful for narrative context, but they may distort event counts if they are treated the same way as individual projects.

### 4.2 Housing Pipeline Dataset

In [6]:
# ── Housing pipeline: basic shape and types ──
housing_raw = pd.read_csv('../data/housing_pipeline_2026.csv', encoding='utf-8-sig')

print("=== HOUSING PIPELINE ===")
print(f"Shape: {housing_raw.shape}")
print()
print("Columns and types:")
print(housing_raw.dtypes.to_string())
print()
print("Missing values per column:")
print(housing_raw.isnull().sum().to_string())
print()
print("Key columns:")
print(housing_raw[['project','units_low','units_high','phase_status','confidence','source_quality']].to_string())

=== HOUSING PIPELINE ===
Shape: (8, 18)

Columns and types:
project                   object
location                  object
units_low                  int64
units_high                 int64
unit_type                 object
phase_status              object
type                      object
row_type                  object
completion_window         object
completion_year_low        int64
completion_year_high       int64
confidence                object
include_in_model            bool
model_exclusion_reason    object
source_quality            object
primary_source_id         object
secondary_source_id       object
notes                     object

Missing values per column:
project                   0
location                  0
units_low                 0
units_high                0
unit_type                 0
phase_status              0
type                      0
row_type                  0
completion_window         0
completion_year_low       0
completion_year_high      0
confidence

#### 5 observations about the housing pipeline dataset:  

1. The dataset contains **8 rows and 18 columns**, which confirms that this is a small, curated project-level dataset focused on major downtown housing developments rather than a full inventory of all residential activity.

2. The unit fields are already numeric. Both **`units_low`** and **`units_high`** are stored as integers, which makes it easy to calculate midpoint estimates and compare project scale across developments.

3. The target-style delivery field, **`phase_status`**, has three clear categories: **Completed**, **Under Construction**, and **Planned**. This makes the dataset suitable for both descriptive analysis and a simple classification workflow.

4. The dataset includes built-in quality and uncertainty fields. **`confidence`** captures how certain the project estimate is, while **`source_quality`** records how strongly each row is supported by public sourcing. These are useful because not all future-oriented housing entries are equally certain.

5. One row is clearly different from the others: **“Other downtown infill / small sites (aggregate)”** is an aggregate estimate rather than a named individual project. That is why the dataset includes an **`include_in_model`** flag and a **`model_exclusion_reason`** field, allowing that row to be separated from the model-ready project set.

### 4.3 Business Licenses Dataset

In [ ]:
import pandas as pd
import os

print("Files in ../data:")
print(os.listdir('../data'))
print()


biz_peek = pd.read_csv('../data/Business_Licenses_20260404.csv', encoding='cp1252', low_memory=False)
downtown_peek = biz_peek[biz_peek['Community Characterization Area'] == 'Downtown']

print("=== BUSINESS LICENSES (downtown only) ===")
print(f"Total file rows: {len(biz_peek)}")
print(f"Downtown rows:   {len(downtown_peek)}")
print()

print("Columns:")
print(list(downtown_peek.columns))
print()

print("Status values:")
print(downtown_peek['Status'].value_counts().to_string())
print()

print("Issue Date sample (raw format):")
print(downtown_peek['Issue Date'].head(3).to_string())

Files in ../data:
['Business_Licenses_20260404.csv', 'dataset.csv', 'downtown_wpg_gantt_sourced_2026.csv', 'downtown_wpg_sources_2026.csv', 'housing_pipeline_2026.csv', 'winnipeg_downtown_projects_2026.csv', 'Winnipeg_Synthetic_License_Model_2010_2024.xlsx']

=== BUSINESS LICENSES (downtown only) ===
Total file rows: 7313
Downtown rows:   2231

Columns:
['Folder Type', 'Folder Description', 'Subdescription', 'Trade Name', 'Address', 'Issue Date', 'Expiry Date', 'Status', 'Neighbourhood Number', 'Neighbourhood Name', 'Electoral Ward', 'Community Characterization Area', 'Location']

Status values:
Status
Closed (L)          1538
Issued               621
Ceased Operation      41
Cancelled             28
Vacant                 2
Review (L)             1

Issue Date sample (raw format):
0    2023 May 10 12:00:00 AM
1    2022 May 10 12:00:00 AM
2    2023 May 10 12:00:00 AM


#### 5 observations about the business license dataset:

1. The full business license file contains **7,313 rows**, of which **2,231** are in the **Downtown** community characterization area. This means the downtown subset is large enough to support trend analysis for recent years.

2. The dataset includes useful business activity fields such as **Folder Type**, **Folder Description**, **Trade Name**, **Issue Date**, **Expiry Date**, **Status**, and location-related columns. These fields make it possible to examine status patterns, timing, and sector-level activity.

3. The **Status** field is heavily dominated by **Closed (L)** entries (**1,538 rows**), followed by **Issued** (**621 rows**). This suggests that the downtown subset includes a large number of inactive or ended licenses and will need careful interpretation when used as a business activity signal.

4. The **Issue Date** field is still stored as text in its raw format, for example **“2023 May 10 12:00:00 AM”**, so it needs to be parsed into datetime format before extracting years or building time-based charts.

5. **This dataset is useful for measuring **recent downtown business conditions**, but it does not provide full historical coverage back to `2010`. That limitation is why the notebook uses a reconstructed pre-2021 business activity series rather than relying on open data alone for the entire study period.**

### 4.4 Sources Dataset

In [10]:
# Load the source provenance registry
sources = pd.read_csv('../data/downtown_wpg_sources_2026.csv', encoding='utf-8-sig')

print(f"Total sources: {len(sources)}")
print(f"  Core inputs (directly feed model): {sources['usage_type'].eq('core_input').sum()}")
print(f"  Model-ready:                       {sources['model_ready'].sum()}")
print(f"  Corroboration sources:             {sources['is_corroboration'].sum()}")

print("\nBy source category:")
print(sources['source_category'].value_counts().to_string())

Total sources: 53
  Core inputs (directly feed model): 15
  Model-ready:                       22
  Corroboration sources:             9

By source category:
source_category
news           28
market_data    13
official        9
background      3


#### 5 observations about the source registry:

1. The project source registry contains **53 total sources**, which shows that the notebook is built from a broad multi-source evidence base rather than a single dataset.

2. Of those 53 sources, **15 are core inputs** that directly feed the analysis or model construction, while **22 are marked model-ready**. This shows that not every collected source is treated equally in the final analytical workflow.

3. The registry also includes **9 corroboration sources**, which are useful for validating dates, costs, or interpretations even when they do not directly drive a chart or model input.

4. The source mix is dominated by **news sources (28)**, followed by **market data (13)** and **official sources (9)**. This reflects the reality of the topic: many structural downtown events are documented through reporting and announcements rather than through a single official administrative dataset.

5. Because the source base is mixed, source quality and evidence labeling are especially important. The registry supports the notebook’s layered design by helping distinguish directly observed facts from reconstructed, inferred, or scenario-based elements.

> **New sources added (Jan 2026):**
> - CBC News. "St. Charles Hotel slated to become 11-storey residential tower." Jan 27, 2026. `[strong, L1, core_input]` — https://www.cbc.ca/news/canada/manitoba/centreventure-winnipeg-downtown-housing-heritage-buildings-9.7063117
> - chrisd.ca. "Nearly 300 New Housing Units Planned for Downtown Winnipeg." Jan 27, 2026. `[moderate, L1, corroboration]` — https://www.chrisd.ca/2026/01/27/downtown-winnipeg-housing-units-redevelopment-centreventure/
> - Residents of the Exchange District. "Towering High Over Maw's Garage." Apr 17, 2025. `[moderate, background]` — https://www.residentsoftheexchangedistrict.ca/post/towering-high-over-maws-garage


### 4.5 Synthetic Business Activity Datase

In [15]:
# ── Synthetic business activity model: documentation sheet overview ──
synthetic_doc = pd.read_excel('../data/Winnipeg_Synthetic_License_Model_2010_2024.xlsx')

print("=== SYNTHETIC BUSINESS ACTIVITY MODEL (DOCUMENTATION SHEET) ===")
print(synthetic_doc.to_string(index=False))
print()

print("Columns and types:")
print(synthetic_doc.dtypes.to_string())
print()

print("Missing values per column:")
print(synthetic_doc.isnull().sum().to_string())
print()

print("First 5 rows:")
print(synthetic_doc.head().to_string(index=False))

=== SYNTHETIC BUSINESS ACTIVITY MODEL (DOCUMENTATION SHEET) ===
Downtown Winnipeg Business License                                                               Synthetic Continuity Model 2010–2024
                               NaN                                                                                                NaN
                           PURPOSE       Gap-fill reconstruction for missing municipal license records 2010–2021. NOT real City data.
                       METHODOLOGY     Calibrated using CBRE vacancy series, BIZ reports, Statistics Canada census, and known events.
                        CONFIDENCE          High for 2022–2024 (open data anchor). Medium for 2015–2021. Low/synthetic for 2010–2014.
                      KNOWN EVENTS Portage Place decline ~2013+, COVID shock 2020, Hudson's Bay closure 2023, remote work shift 2020+
                          DATA GAP                  City of Winnipeg retention policy = 10 years. Pre-2014 records legally destroyed

#### 5 observations about the synthetic business activity model file:

1. The workbook functions as a **documentation sheet rather than a row-level analytical dataset**. Instead of yearly business records, it contains notes describing the purpose, methodology, confidence, and limitations of the synthetic reconstruction.

2. Both columns are stored as **text (`object`)**, which confirms that this sheet is descriptive and not directly usable as a numeric time-series table.

3. The file clearly states that the synthetic continuity model is a **gap-fill reconstruction for missing municipal license records from 2010 to 2021** and is **not real City data**. This is an important methodological limitation and should stay visible throughout the notebook.

4. The built-in confidence note is especially useful because it distinguishes stronger and weaker parts of the reconstruction: **high confidence for 2022–2024**, **medium confidence for 2015–2021**, and **low/synthetic confidence for 2010–2014**.

5. The sheet also documents the reason the synthetic model was needed in the first place: a **10-year City of Winnipeg records retention limit** created a historical data gap, so the synthetic series was used as a transparent workaround rather than a substitute for official historical records.

## 5. Data Cleaning

This section documents every cleaning step and explains *why* it was needed — not just what was done.

### 5.1 Gantt Dataset Cleaning

In [16]:
# ── CLEAN THE GANTT DATASET ──
import pandas as pd

gantt = pd.read_csv(
    '../data/downtown_wpg_gantt_sourced_2026.csv',
    parse_dates=['start_date', 'end_date']
)

# Step 1: Fill missing end dates
# WHY: In-progress projects have no end date. We fill with the data collection date
# so that duration calculations work. This is NOT a completion date — it means
# "still running as of April 2026."
TODAY = pd.Timestamp('2026-04-13')
missing_before = gantt['end_date'].isna().sum()
gantt['end_date'] = gantt['end_date'].fillna(TODAY)
print(f"Filled {missing_before} missing end dates -> {TODAY.date()}")

# Step 2: Compute duration and extract years as integers
# WHY: Raw datetime objects aren't useful for charts — we need plain numbers.
gantt['duration_days'] = (gantt['end_date'] - gantt['start_date']).dt.days
gantt['year_start'] = gantt['start_date'].dt.year
gantt['year_end'] = gantt['end_date'].dt.year

# Step 3: Assign a phase label based on start year
# WHY: The dataset has no phase column. We derive it from year_start using
# researcher-defined boundaries based on known downtown events.
def assign_phase(y):
    if y < 2015:
        return 'Pre-2015 establishment'
    elif y <= 2020:
        return '2015–2020 transition'
    else:
        return '2020+ restructuring'

gantt['phase'] = gantt['year_start'].apply(assign_phase)

# Step 4: Map each category to Growth or Transition
# WHY: We need a binary signal to count events in phase composition charts.
impact_map = {
    'Growth': 'Growth',
    'Infrastructure': 'Growth',
    'Adaptive Reuse': 'Growth',
    'Transition': 'Transition',
    'Policy': 'Transition',
}
gantt['impact'] = gantt['category'].map(impact_map).fillna('Transition')

# Step 5: Create a completion flag
# WHY: The Gantt chart uses solid vs. hatched bars to show completion status.
completed_kw = ['Completed', 'Adopted', 'Demolished', 'Active']
gantt['is_done'] = gantt['status'].apply(
    lambda s: any(kw in str(s) for kw in completed_kw)
)

print(f"\nGantt cleaned: {len(gantt)} rows, {gantt.isnull().sum().sum()} remaining nulls in key columns")

Filled 11 missing end dates -> 2026-04-13

Gantt cleaned: 30 rows, 6 remaining nulls in key columns


#### Cleaning Summary

- The Gantt dataset was cleaned successfully and is now ready for timeline-based analysis. A total of **11 missing `end_date` values** were filled with **April 13, 2026**, which represents the data collection date rather than a true completion date. This step allows ongoing projects to remain in the analysis instead of being dropped during duration calculations.  

- After cleaning, the dataset contains **30 rows**. New fields were added to support later analysis, including project duration in days, extracted start and end years, a phase label based on start year, a simplified impact grouping, and a completion flag for chart styling.  

- There are still **6 remaining null values** in the dataset, but these are not in the main fields needed for timeline construction. The core date and classification columns are now usable, so the dataset is in good shape for the event timeline and phase-based analysis.

### 5.2 Housing Pipeline Cleaning

In [17]:
# ── CLEAN THE HOUSING PIPELINE ──
import pandas as pd

housing = pd.read_csv('../data/housing_pipeline_2026.csv', encoding='utf-8-sig')

# Step 1: Compute midpoint estimate
# WHY: Unit counts are ranges (low/high). The midpoint gives one representative
# number for charts and as a model feature.
housing['units_mid'] = (housing['units_low'] + housing['units_high']) / 2

# Step 2: Separate model-ready rows from the aggregate infill row
# WHY: The "Other infill" row is an unverifiable aggregate. Treating it the same
# as named, sourced projects would distort the pipeline totals and model.
housing_model = housing[housing['include_in_model']].copy()

print(f"Total rows:      {len(housing)}")
print(f"Model-ready:     {len(housing_model)}")
print(f"Excluded row:    {housing[~housing['include_in_model']]['project'].values}")
print(f"\nphase_status values: {sorted(housing['phase_status'].unique())}")
print(f"\nconfidence values:   {sorted(housing['confidence'].unique())}")

Total rows:      8
Model-ready:     7
Excluded row:    ['Other downtown infill / small sites (aggregate)']

phase_status values: ['Completed', 'Planned', 'Under Construction']

confidence values:   ['high', 'low_medium', 'medium', 'medium_high']


#### Cleaning Summary

The housing pipeline dataset contains **8 total rows**, of which **7 were kept as model-ready projects**. One row, **“Other downtown infill / small sites (aggregate)”**, was excluded because it represents a grouped estimate rather than a clearly identifiable, source-specific project. Keeping it out of the model-ready set helps preserve project-level consistency and avoids mixing named developments with broad approximations.

The cleaned dataset includes three project status categories: **Completed**, **Planned**, and **Under Construction**. This makes the file suitable for both descriptive analysis and a simple classification example later in the notebook.

The confidence field also shows different certainty levels, ranging from **high** to **low_medium**. That variation matters because the housing pipeline is forward-looking, and not all projects are equally certain. Overall, the cleaning step produced a small but usable project-level dataset for analyzing downtown residential delivery and future supply.

### 5.3 Business License Dataset Cleaning

In [19]:
# ── CLEAN THE BUSINESS LICENSE DATA ──

biz_raw = pd.read_csv('../data/Business_Licenses_20260404.csv',
                      encoding='cp1252', low_memory=False)

# Step 1: Convert the date string to a proper datetime object
# WHY: The format "2023 May 10 12:00:00 AM" is not parseable by default.
# The explicit format string tells pandas exactly how to read it.
biz_raw['Issue_DT'] = pd.to_datetime(
    biz_raw['Issue Date'], format='%Y %b %d %H:%M:%S %p', errors='coerce'
)
biz_raw['year'] = biz_raw['Issue_DT'].dt.year

# Step 2: Filter to downtown only
# WHY: The open data file covers all of Winnipeg. We only need the downtown area.
biz = biz_raw[biz_raw['Community Characterization Area'] == 'Downtown'].copy()

# Check how many dates failed to parse
failed = biz_raw['Issue_DT'].isna().sum()
print(f"Date parsing failures: {failed}")
print(f"Downtown records kept: {len(biz)}")
print(f"Year range:            {int(biz['year'].min())} – {int(biz['year'].max())}")

Date parsing failures: 0
Downtown records kept: 2231
Year range:            2021 – 2026


#### Cleaning Summary

The downtown business license dataset cleaned successfully with **0 date parsing failures**, which means the issue dates were converted reliably into datetime format. After filtering the full file to the study area, **2,231 downtown records** were retained for analysis.

The cleaned downtown subset covers a **year range from 2021 to 2026**, which provides a recent multi-year window for examining business activity, closures, and license status patterns. This is useful for measuring current conditions, but it also confirms one of the project’s core limitations: the official open data does not extend far enough back to analyze the full 2010–2026 structural transition on its own.

Overall, the business license data is now ready for time-based analysis of recent downtown conditions. It serves as the strongest directly observed business activity source in the notebook, while earlier years still rely on reconstructed trend estimates.

### Overall Cleaning Summary

| Dataset | Problem | Fix | Why |
|---|---|---|---|
| Gantt | Missing `end_date` values for in-progress rows | Filled missing values with **April 13, 2026** | Allows duration calculations while clearly treating these as still ongoing at the time of data collection |
| Gantt | No `phase`, `impact`, or `is_done` fields | Derived new columns from existing date, category, and status fields | Required to support phase analysis, impact grouping, and Gantt chart styling |
| Housing | Unit counts stored as ranges | Computed a midpoint estimate, `units_mid` | Provides a single comparable numeric value for charts and simple modeling |
| Housing | One aggregate row mixed with named project rows | Separated into `housing_model` and excluded the aggregate row from project-level modeling | Prevents unverifiable grouped estimates from distorting project-level analysis |
| Business licenses | Dates stored as non-standard strings | Parsed dates using an explicit format string | Makes year extraction and time-based trend analysis possible |
| Business licenses | Citywide records included in the raw file | Filtered to `Downtown` records only | Keeps the dataset aligned with the project’s downtown-only scope |

Overall, the cleaning process focused on making each dataset analytically usable while preserving transparency about assumptions, derived fields, and excluded records. These steps were necessary to support consistent comparison across the three sources and to ensure that later visualizations and interpretations were based on usable, defensible inputs.